# Fixed-change iteration test

This notebook uses the same pocket candidate search logic as the existing genetic algorithm notebook, but it changes the stopping rule:

- Start each repeat with a random gene set.
- For exactly 100 proposal changes, randomly change 1-9 loci.
- Keep the proposal only when it improves the similarity score; otherwise keep the old set.
- Repeat this 100 times and count how often the kept set reaches or exceeds the causal gene similarity.

Run modes:

- One disease: set `DISEASE_ID = "EFO_0000616"` and optionally set `FOLDER_NAME`.
- Whole folder: set `FOLDER_NAME = "3_gene_related_disease"` and `DISEASE_ID = None`.


In [178]:
from pathlib import Path

PROJECT_ROOT = Path("/Users/miasmacbook/Desktop/KCL/6-months_project")
SCRIPT_PATH = PROJECT_ROOT / "script" / "rerun_pocket_models_from_separate_disease.py"
GLOBAL_SUMMARY_PATH = PROJECT_ROOT / "Similariy" / "TPM_disease_avg_similarity_summary.csv"
OUTPUT_ROOT = PROJECT_ROOT / "pocket_model_iterations"

# Set these before running.
# To run one disease, set DISEASE_ID to that disease ID.
# To run every disease under a folder, set DISEASE_ID = None and FOLDER_NAME = "3_gene_related_disease".
DISEASE_ID = None
FOLDER_NAME = "121_gene_related_disease"

N_REPEATS = 100
N_CHANGES_PER_REPEAT = 100
SEED_START = 42
MAX_GENES_TO_CHANGE = 9
REQUIRE_UNIQUE_GENES = False

COMBINED_SUMMARY_CSV_NAME = "100_change_iterations_combined_summary.csv"


In [179]:
from __future__ import annotations

import itertools
import math
import random
import runpy

import matplotlib.pyplot as plt
import pandas as pd

script_globals = runpy.run_path(str(SCRIPT_PATH), run_name="iteration_helpers")
choose_initial_random_selection = script_globals["choose_initial_random_selection"]
compute_average_similarity = script_globals["compute_average_similarity"]
discover_disease_inputs = script_globals["discover_disease_inputs"]
load_global_summary = script_globals["load_global_summary"]
load_saved_round_summary = script_globals["load_saved_round_summary"]
load_subset_matrix = script_globals["load_subset_matrix"]
metrics_are_better = script_globals["metrics_are_better"]
propose_selection = script_globals["propose_selection"]


def get_disease_input(project_root: Path, disease_id: str, folder_name: str | None = None):
    folder_names = [folder_name] if folder_name else []
    disease_inputs = discover_disease_inputs(
        project_root=project_root,
        wanted_folder_names=folder_names,
        wanted_disease_ids=[disease_id],
    )

    if not disease_inputs:
        raise FileNotFoundError(f"Could not find required inputs for disease ID: {disease_id}")

    if len(disease_inputs) > 1:
        matches = [f"{item.folder_name}/{item.disease_id}" for item in disease_inputs]
        raise ValueError(
            "More than one matching disease folder was found. "
            f"Set FOLDER_NAME explicitly. Matches: {matches}"
        )

    return disease_inputs[0]


def get_disease_inputs_for_run(project_root: Path, disease_id: str | None, folder_name: str | None = None):
    folder_names = [folder_name] if folder_name else []
    disease_ids = [disease_id] if disease_id else []
    disease_inputs = discover_disease_inputs(
        project_root=project_root,
        wanted_folder_names=folder_names,
        wanted_disease_ids=disease_ids,
    )

    if not disease_inputs:
        if disease_id:
            raise FileNotFoundError(f"Could not find required inputs for disease ID: {disease_id}")
        raise FileNotFoundError(f"Could not find required disease inputs under folder: {folder_name}")

    if disease_id and len(disease_inputs) > 1:
        matches = [f"{item.folder_name}/{item.disease_id}" for item in disease_inputs]
        raise ValueError(
            "More than one matching disease folder was found. "
            f"Set FOLDER_NAME explicitly. Matches: {matches}"
        )

    return disease_inputs


def resolve_baseline_score(disease_id: str, saved_round_summary: dict):
    global_summary = load_global_summary(GLOBAL_SUMMARY_PATH)
    summary_row = global_summary.get(disease_id)
    if summary_row:
        value_text = (summary_row.get("avg_pairwise_similarity") or "").strip()
        if value_text:
            return float(value_text)
    return saved_round_summary.get("saved_baseline_score")


def candidate_display_name(candidate) -> str:
    return (
        candidate.neighbor_symbol
        or candidate.matrix_gene_id
        or candidate.candidate_label
        or candidate.seed_target_id
    )


def build_final_set_string(seed_target_ids, selection) -> str:
    parts = []
    for seed_target_id in seed_target_ids:
        candidate = selection[seed_target_id]
        parts.append(f"{seed_target_id}->{candidate_display_name(candidate)}")
    return " | ".join(parts)


def collect_missing_genes(seed_target_ids, selection, similarity_matrix) -> str:
    missing = set()
    selected_candidates = [selection[seed_target_id] for seed_target_id in seed_target_ids]

    for candidate in selected_candidates:
        if not candidate.matrix_gene_id:
            missing.add(candidate_display_name(candidate))

    for left_candidate, right_candidate in itertools.combinations(selected_candidates, 2):
        left_gene = left_candidate.matrix_gene_id
        right_gene = right_candidate.matrix_gene_id

        if not left_gene or not right_gene:
            missing.add(candidate_display_name(left_candidate))
            missing.add(candidate_display_name(right_candidate))
            continue

        value = similarity_matrix.get(left_gene, {}).get(right_gene)
        if value is None:
            value = similarity_matrix.get(right_gene, {}).get(left_gene)

        if value is None:
            missing.add(candidate_display_name(left_candidate))
            missing.add(candidate_display_name(right_candidate))

    return " | ".join(sorted(item for item in missing if item))


def run_fixed_change_repeat(
    disease_input,
    similarity_matrix,
    baseline_score: float,
    rng_seed: int,
    n_changes: int,
    max_genes_to_change: int,
    require_unique_genes: bool,
):
    rng = random.Random(rng_seed)
    changeable_seed_target_ids = [
        seed_target_id
        for seed_target_id in disease_input.seed_target_ids
        if len(disease_input.pockets[seed_target_id]) > 1
    ]

    current_selection = choose_initial_random_selection(
        seed_target_ids=disease_input.seed_target_ids,
        pockets=disease_input.pockets,
        rng=rng,
        require_unique_genes=require_unique_genes,
    )
    current_metrics = compute_average_similarity(list(current_selection.values()), similarity_matrix)
    current_score = current_metrics["avg_similarity"]
    initial_score = current_score

    initial_hit = current_score is not None and current_score >= baseline_score
    hit_baseline = initial_hit
    first_hit_change = 0 if initial_hit else None
    accepted_changes = 0
    attempted_changes = 0
    best_change = 0

    if not changeable_seed_target_ids:
        return {
            "seed": rng_seed,
            "attempted_changes": 0,
            "accepted_changes": 0,
            "hit_causal_similarity": hit_baseline,
            "initial_hit_causal_similarity": initial_hit,
            "first_hit_change": first_hit_change,
            "initial_score": current_score,
            "final_score": current_score,
            "best_score": current_score,
            "best_change": best_change,
            "final_set_of_gene": build_final_set_string(disease_input.seed_target_ids, current_selection),
            "missing_gene": collect_missing_genes(disease_input.seed_target_ids, current_selection, similarity_matrix),
            "stop_reason": "no_changeable_pockets",
        }

    while attempted_changes < n_changes:
        n_to_change = rng.randint(1, min(max_genes_to_change, len(changeable_seed_target_ids)))
        chosen_seed_target_ids = rng.sample(changeable_seed_target_ids, n_to_change)

        proposal_selection, previous_candidates, proposed_candidates = propose_selection(
            seed_target_ids=disease_input.seed_target_ids,
            pockets=disease_input.pockets,
            current_selection=current_selection,
            chosen_seed_target_ids=chosen_seed_target_ids,
            rng=rng,
            require_unique_genes=require_unique_genes,
        )
        if proposal_selection is None:
            continue

        attempted_changes += 1
        proposal_metrics = compute_average_similarity(list(proposal_selection.values()), similarity_matrix)

        if metrics_are_better(proposal_metrics, current_metrics):
            current_selection = proposal_selection
            current_metrics = proposal_metrics
            current_score = current_metrics["avg_similarity"]
            accepted_changes += 1
            best_change = attempted_changes

        if current_score is not None and current_score >= baseline_score:
            hit_baseline = True
            if first_hit_change is None:
                first_hit_change = attempted_changes

    return {
        "seed": rng_seed,
        "attempted_changes": attempted_changes,
        "accepted_changes": accepted_changes,
        "hit_causal_similarity": hit_baseline,
        "initial_hit_causal_similarity": initial_hit,
        "first_hit_change": first_hit_change if first_hit_change is not None else "",
        "initial_score": initial_score,
        "final_score": current_score,
        "best_score": current_score,
        "best_change": best_change,
        "final_set_of_gene": build_final_set_string(disease_input.seed_target_ids, current_selection),
        "missing_gene": collect_missing_genes(disease_input.seed_target_ids, current_selection, similarity_matrix),
        "stop_reason": "fixed_change_limit_reached",
    }


def build_distribution_line(values: pd.Series, n_points: int = 256):
    samples = [float(value) for value in values]
    if len(samples) < 2:
        return None

    mean_value = sum(samples) / len(samples)
    variance = sum((value - mean_value) ** 2 for value in samples) / (len(samples) - 1)
    if variance <= 0:
        return None

    std_dev = math.sqrt(variance)
    bandwidth = 1.06 * std_dev * (len(samples) ** (-1 / 5))
    if not math.isfinite(bandwidth) or bandwidth <= 0:
        return None

    x_min = min(samples)
    x_max = max(samples)
    if math.isclose(x_min, x_max):
        return None

    padding = max((x_max - x_min) * 0.08, bandwidth * 2)
    x_start = x_min - padding
    x_stop = x_max + padding
    x_values = [
        x_start + ((x_stop - x_start) * index / (n_points - 1))
        for index in range(n_points)
    ]

    coefficient = 1 / (len(samples) * bandwidth * math.sqrt(2 * math.pi))
    density_values = []
    for x_value in x_values:
        density = coefficient * sum(
            math.exp(-0.5 * ((x_value - sample) / bandwidth) ** 2)
            for sample in samples
        )
        density_values.append(density)

    return x_values, density_values


def create_iteration_distribution_plot(results_df: pd.DataFrame, output_path: Path, disease_id: str, baseline_score: float):
    valid_scores = pd.to_numeric(results_df["final_score"], errors="coerce").dropna()
    if valid_scores.empty:
        raise ValueError("No valid final scores were available for plotting.")

    bins = min(20, max(10, int(math.sqrt(len(valid_scores)))))
    mean_score = float(valid_scores.mean())
    max_score = float(valid_scores.max())
    distribution_line = build_distribution_line(valid_scores)

    fig, ax = plt.subplots(figsize=(10, 6))
    fig.patch.set_facecolor("white")
    ax.set_facecolor("white")

    ax.hist(valid_scores, bins=bins, color="#1d4ed8", edgecolor="white", alpha=0.82)
    ax.axvline(
        mean_score,
        color="#b91c1c",
        linewidth=2.0,
        linestyle="--",
        label=f"Mean final score: {mean_score:.4f}",
    )
    ax.axvline(
        max_score,
        color="#ea580c",
        linewidth=2.1,
        linestyle="-",
        label=f"Best final score: {max_score:.4f}",
    )
    ax.axvline(
        float(baseline_score),
        color="#16a34a",
        linewidth=1.9,
        linestyle="-.",
        label=f"Causal gene similarity: {float(baseline_score):.4f}",
    )

    density_axis = None
    if distribution_line is not None:
        x_values, density_values = distribution_line
        density_axis = ax.twinx()
        density_axis.set_facecolor("none")
        density_axis.plot(
            x_values,
            density_values,
            color="#0f766e",
            linewidth=2.2,
            label="Distribution line",
        )
        density_axis.set_ylabel("Density", fontsize=13, color="#0f766e")
        density_axis.tick_params(axis="y", colors="#0f766e")
        density_axis.spines["top"].set_visible(False)

    ax.set_title(f"{disease_id} final scores across 100 fixed-change repeats", fontsize=16, pad=12)
    ax.set_xlabel("Final score after 100 changes", fontsize=13)
    ax.set_ylabel("Number of repeats", fontsize=13)
    ax.grid(True, axis="y", color="#cbd5e1", linewidth=0.8, alpha=0.8)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    legend_handles, legend_labels = ax.get_legend_handles_labels()
    if density_axis is not None:
        density_handles, density_labels = density_axis.get_legend_handles_labels()
        legend_handles.extend(density_handles)
        legend_labels.extend(density_labels)
    ax.legend(legend_handles, legend_labels, frameon=False, loc="best")

    output_path.parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(output_path, dpi=150)
    plt.close(fig)


def summarize_iteration_results(results_df: pd.DataFrame, disease_id: str, folder_name: str, baseline_score: float) -> pd.DataFrame:
    hit_count = int(results_df["hit_causal_similarity"].sum())
    initial_hit_count = int(results_df["initial_hit_causal_similarity"].sum())
    final_scores = pd.to_numeric(results_df["final_score"], errors="coerce")
    first_hits = pd.to_numeric(results_df["first_hit_change"], errors="coerce").dropna()

    return pd.DataFrame([
        {
            "disease_id": disease_id,
            "folder_name": folder_name,
            "causal_gene_similarity": baseline_score,
            "n_repeats": len(results_df),
            "changes_per_repeat": N_CHANGES_PER_REPEAT,
            "hit_count": hit_count,
            "hit_rate": hit_count / len(results_df) if len(results_df) else 0,
            "initial_hit_count": initial_hit_count,
            "hit_after_change_count": hit_count - initial_hit_count,
            "mean_final_score": float(final_scores.mean()) if not final_scores.empty else None,
            "median_final_score": float(final_scores.median()) if not final_scores.empty else None,
            "max_final_score": float(final_scores.max()) if not final_scores.empty else None,
            "mean_first_hit_change": float(first_hits.mean()) if not first_hits.empty else None,
        }
    ])


In [180]:
def run_iterations_for_disease(disease_input):
    subset_matrix = load_subset_matrix(disease_input.matrix_subset_csv)
    saved_round_summary = load_saved_round_summary(disease_input.saved_rounds_csv)
    baseline_score = resolve_baseline_score(disease_input.disease_id, saved_round_summary)
    if baseline_score is None:
        raise ValueError(f"Could not determine causal gene similarity for {disease_input.disease_id}")

    output_dir = OUTPUT_ROOT / disease_input.folder_name / disease_input.disease_id
    output_dir.mkdir(parents=True, exist_ok=True)

    repeat_rows = []
    for repeat_index in range(N_REPEATS):
        seed = SEED_START + repeat_index
        repeat_rows.append(
            run_fixed_change_repeat(
                disease_input=disease_input,
                similarity_matrix=subset_matrix,
                baseline_score=baseline_score,
                rng_seed=seed,
                n_changes=N_CHANGES_PER_REPEAT,
                max_genes_to_change=MAX_GENES_TO_CHANGE,
                require_unique_genes=REQUIRE_UNIQUE_GENES,
            )
        )

    results_df = pd.DataFrame(repeat_rows)
    summary_df = summarize_iteration_results(
        results_df,
        disease_id=disease_input.disease_id,
        folder_name=disease_input.folder_name,
        baseline_score=baseline_score,
    )

    repeat_result_csv_name = f"{disease_input.disease_id}_100_change_iterations.csv"
    summary_csv_name = f"{disease_input.disease_id}_100_change_iterations_summary.csv"
    plot_name = f"{disease_input.disease_id}_100_change_iterations_distribution.png"

    result_csv_path = output_dir / repeat_result_csv_name
    summary_csv_path = output_dir / summary_csv_name
    plot_path = output_dir / plot_name
    results_df.to_csv(result_csv_path, index=False)
    summary_df.to_csv(summary_csv_path, index=False)
    create_iteration_distribution_plot(results_df, plot_path, disease_input.disease_id, baseline_score)

    return {
        "disease_input": disease_input,
        "baseline_score": baseline_score,
        "results_df": results_df,
        "summary_df": summary_df,
        "result_csv_path": result_csv_path,
        "summary_csv_path": summary_csv_path,
        "plot_path": plot_path,
    }


disease_inputs = get_disease_inputs_for_run(PROJECT_ROOT, DISEASE_ID, FOLDER_NAME)
print(f"Diseases to run: {len(disease_inputs)}")

run_outputs = []
for index, disease_input in enumerate(disease_inputs, start=1):
    print(f"[{index}/{len(disease_inputs)}] Running {disease_input.folder_name}/{disease_input.disease_id}")
    run_outputs.append(run_iterations_for_disease(disease_input))

combined_summary_df = pd.concat(
    [item["summary_df"] for item in run_outputs],
    ignore_index=True,
)

if FOLDER_NAME:
    combined_summary_dir = OUTPUT_ROOT / FOLDER_NAME
elif len(disease_inputs) == 1:
    combined_summary_dir = OUTPUT_ROOT / disease_inputs[0].folder_name
else:
    combined_summary_dir = OUTPUT_ROOT
combined_summary_dir.mkdir(parents=True, exist_ok=True)
combined_summary_path = combined_summary_dir / COMBINED_SUMMARY_CSV_NAME
combined_summary_df.to_csv(combined_summary_path, index=False)

print(f"Combined summary CSV: {combined_summary_path}")
for item in run_outputs:
    disease_input = item["disease_input"]
    summary = item["summary_df"].iloc[0]
    print(
        f"{disease_input.folder_name}/{disease_input.disease_id}: "
        f"hit {int(summary['hit_count'])}/{int(summary['n_repeats'])} "
        f"({float(summary['hit_rate']):.2%}); graph: {item['plot_path']}"
    )

combined_summary_df


Diseases to run: 1
[1/1] Running 121_gene_related_disease/EFO_0004995
Combined summary CSV: /Users/miasmacbook/Desktop/KCL/6-months_project/pocket_model_iterations/121_gene_related_disease/100_change_iterations_combined_summary.csv
121_gene_related_disease/EFO_0004995: hit 100/100 (100.00%); graph: /Users/miasmacbook/Desktop/KCL/6-months_project/pocket_model_iterations/121_gene_related_disease/EFO_0004995/EFO_0004995_100_change_iterations_distribution.png


,disease_id,folder_name,causal_gene_similarity,n_repeats,changes_per_repeat,hit_count,hit_rate,initial_hit_count,hit_after_change_count,mean_final_score,median_final_score,max_final_score,mean_first_hit_change
0,EFO_0004995,121_gene_related_disease,0.201218,100,100,100,1.0,0,100,0.260575,0.259878,0.300111,34.6
